# 06 — TaskGroups, Cancellation, and Timeouts

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
import asyncio

# Provided for this lesson.
from collections.abc import Awaitable, Callable, Iterable

from lesson import Exercises, ___, run

check = Exercises('advanced/06', {1: 'f37476fd617f54d6', 2: '8dd5214b56add723', 3: 'e8fe3c4579fa64f2', 4: '8d566ac8dcbc131a', 5: 'db788026a98966b8', 6: '0e445f8198a09df4', 7: '2cfc009f5644fdd4', 8: '011c95d967566533'}, practice='advanced_02_concurrency_resilience', independent='run_bounded')

## 1. TaskGroup

- `async with asyncio.TaskGroup() as group:` then `group.create_task(coro)`. The block waits for every task.
- If one task fails, the others are cancelled and the errors are raised together as an `ExceptionGroup`.
- Keep the task objects to read results afterwards: `task.result()`.

In [ ]:
import asyncio

async def double(x):
    await asyncio.sleep(0.01 * (3 - x))
    return x * 2

async def main():
    async with asyncio.TaskGroup() as group:
        tasks = [group.create_task(double(x)) for x in [1, 2, 3]]
    return [t.result() for t in tasks]

print(run(main()))

**Exercise 1.** Collect the results in input order from the tasks.

In [ ]:
import asyncio

async def neg(x):
    await asyncio.sleep(0)
    return -x

async def main():
    async with asyncio.TaskGroup() as group:
        tasks = [group.create_task(neg(x)) for x in [5, 6]]
    return [___ for t in tasks]   # each task's result()

check(1, run(main()))

**Exercise 2.** A task in a `TaskGroup` raises `ValueError`. Which exception type reaches `except*`'s caller? Write its name.

In [ ]:
answer = ___
check(2, answer)

**Exercise 3.** `except* ValueError` catches the matching errors inside the group. Predict the outcome.

In [ ]:
import asyncio

async def bad():
    raise ValueError("boom")

async def main():
    outcome = "not caught"
    try:
        async with asyncio.TaskGroup() as group:
            group.create_task(bad())
    except* ValueError:
        outcome = "handled"
    return outcome

predicted = ___   # run(main())
check(3, predicted)

## 2. Timeouts

- `async with asyncio.timeout(seconds):` cancels the block if it runs too long and raises `TimeoutError`.
- Turn a timeout into a value at the boundary: `("timeout", None)` instead of crashing.
- Every network call needs a timeout.

In [ ]:
import asyncio

async def slow():
    await asyncio.sleep(1)
    return "finished"

async def main():
    try:
        async with asyncio.timeout(0.05):
            return ("ok", await slow())
    except TimeoutError:
        return ("timeout", None)

print(run(main()))

**Exercise 4.** A 0.01 s job under a 1 s timeout. Predict the result.

In [ ]:
import asyncio

async def quick():
    await asyncio.sleep(0.01)
    return 7

async def main():
    try:
        async with asyncio.timeout(1):
            return ("ok", await quick())
    except TimeoutError:
        return ("timeout", None)

result = run(main())
predicted = ___
check(4, predicted)

**Exercise 5.** Which exception does `asyncio.timeout` raise when time runs out? Write its name.

In [ ]:
answer = ___
check(5, answer)

## 3. Cancellation, limits, queues

- Cancelling a task raises `CancelledError` inside it; use `finally` to clean up, and don't swallow it.
- `asyncio.Semaphore(limit)` caps how many run at once: `async with sem:` around the work.
- `asyncio.Queue(maxsize=n)` connects producers and consumers; a `None` sentinel tells the consumer to stop.

In [ ]:
import asyncio

async def main():
    sem = asyncio.Semaphore(2)
    running, peak = 0, 0
    async def job(x):
        nonlocal running, peak
        async with sem:
            running += 1
            peak = max(peak, running)
            await asyncio.sleep(0.01)
            running -= 1
            return x
    results = await asyncio.gather(*(job(x) for x in range(5)))
    return results, peak

print(run(main()))

**Exercise 6.** Five jobs with `Semaphore(2)`: what is the most running at once?

In [ ]:
predicted = ___
check(6, predicted)

**Exercise 7.** Stop the consumer at the `None` sentinel.

In [ ]:
import asyncio

async def main():
    queue = asyncio.Queue()
    for item in [1, 2, None]:
        await queue.put(item)
    seen = []
    for _ in range(3):
        item = await queue.get()
        if item is ___:
            break
        seen.append(item)
    return seen

check(7, run(main()))

**Exercise 8.** Does `finally` run when a task is cancelled? Write `True` or `False`.

In [ ]:
answer = ___
check(8, answer)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `with_timeout`

Return `('ok', result)` or `('timeout', None)`.

In [ ]:
async def with_timeout(awaitable: Awaitable, seconds: float) -> tuple[str, object]:
    """Return ('ok', result) or ('timeout', None) using asyncio.timeout()."""
    try:
        async with asyncio.timeout(seconds):
            return ___
    except TimeoutError:
        return ___

In [ ]:
check('with_timeout')

### Independent build — `run_bounded`

Run the async worker on each input, allowing at most limit active workers (limit is positive). Return (results in input order, observed peak active workers). For empty input return ([], 0).

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
async def run_bounded(values: Iterable, worker: Callable[[object], Awaitable], limit: int) -> tuple[list, int]:
    raise NotImplementedError

In [ ]:
check('run_bounded')

### 3. `pipeline`

A producer and consumer joined by a bounded queue and a `None` sentinel.

In [ ]:
async def pipeline(items: Iterable, transform: Callable[[object], object], maxsize: int = 2) -> list:
    """Producer/consumer through a bounded asyncio.Queue; a None sentinel stops the consumer.

    Return transform(item) for each item, in order.
    """
    raise NotImplementedError

In [ ]:
check('pipeline')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** When does `async with asyncio.TaskGroup()` exit?

   - **a)** After the first task finishes
   - **b)** Immediately
   - **c)** After all its child tasks finish

**2.** If one TaskGroup child fails, its siblings are…

   - **a)** Left running
   - **b)** Restarted
   - **c)** Cancelled

**3.** Swallowing CancelledError is…

   - **a)** Recommended
   - **b)** Required
   - **c)** Usually a bug: cancellation should propagate

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/06', {1: '08ee4af13cfa', 2: 'cca8d96d083d', 3: 'd7cf183d1680'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [07 — Threads, Locks, Queues, and Shared State](07_threading_locks_queues.ipynb)